# Florian Ewing, Margaret Keau, John Farnandez, Lauren Connely
# Professor Ix Procopios
# Software Design and Implementation
# Adapted: 2014–2024 BTS Flight Delay Dataset

# Optimum Departure Location & Airline by State Predictor
# (2014–2024 BTS Dataset Adaptation)

This notebook builds a machine learning model that predicts whether a U.S. domestic flight
will depart on time or experience a significant delay, then uses that model to recommend
the best airport and airline combination for a traveler to minimize their delay risk.

The dataset is the **2014–2024 Delayed Flight Data** (`JOHNFF09/2014-2024-delayed-flight-data`)
sourced from Kaggle, which contains over 63 million flights exported directly from the
Bureau of Transportation Statistics (BTS) on-time performance database. Because BTS exports
encode every categorical field as a numeric code, three official BTS lookup tables are fetched
at runtime to decode airport IDs to city/state labels, city market IDs to city names, and
carrier codes to full airline names.

The model uses a **binary classification target**:
- **Class 0** → Flight departed on time or fewer than 15 minutes late
- **Class 1** → Flight departed 15 or more minutes late

Features used for prediction include the airline, departure airport, year, month, day of week,
weekend flag, holiday proximity flag, scheduled departure hour, and the breakdown of delay
causes available in the BTS data (weather, carrier, NAS, and late aircraft delays).
A Random Forest classifier is trained on a stratified 3-million-row sample of the full dataset
and evaluated on a held-out 20% test split.

The recommender accepts a **U.S. state** and departure date from the user, evaluates every
airport and airline combination that has historically operated in that state, and returns
the top 5 combinations with the lowest predicted delay probability. This replaces the
original broad region-based input (e.g. 'West') with state-level granularity so that a
traveler in Washington is not recommended airports in Colorado.

The trained model and label encoders are saved to disk at the end of training so the
predictor can be loaded and used in other environments without retraining.

# Cell 1 — Download Dataset

Downloads the 2014–2024 BTS flight delay dataset from Kaggle using `kagglehub`,
which handles authentication automatically via Colab's built-in Kaggle integration.
Also imports all libraries used throughout the notebook.

In [ ]:
# ===== Cell 1: Download Dataset =====
import os
import pandas as pd
import numpy as np
import kagglehub

dataset_path = kagglehub.dataset_download("JOHNFF09/2014-2024-delayed-flight-data")
print("Downloaded to:", dataset_path)

print("\nFiles:")
for f in sorted(os.listdir(dataset_path)):
    size_mb = os.path.getsize(os.path.join(dataset_path, f)) / 1e6
    print(f"  {f:50s}  {size_mb:7.1f} MB")

# Cell 2 — Fetch BTS Lookup Tables

Fetches three lookup tables directly from the BTS transtats server to decode
the numeric codes stored in the raw dataset into human-readable values.
The airport lookup maps numeric airport IDs to city and state labels.
The city market lookup provides a backup state source for any airports
not resolved by the airport lookup. The carrier lookup maps two-letter
carrier codes to full airline names.

In [ ]:
# ===== Cell 2: Fetch BTS Lookup Tables =====
import requests, io, zipfile

BTS_LOOKUP_URLS = {
    "airport":     "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_NVecbeg_VQ",
    "city_market": "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_PVgl_ZNeXRg_VQ",
    "carrier":     "https://www.transtats.bts.gov/Download_Lookup.asp?Y11x72=Y_haVdhR_PNeeVRef",
}

def fetch_bts_lookup(url):
    """Fetch a BTS lookup table. Handles both plain CSV and zip responses."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    raw = resp.content
    if raw[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            csv_name = [n for n in z.namelist() if n.endswith(".csv")][0]
            with z.open(csv_name) as f:
                return pd.read_csv(f)
    else:
        return pd.read_csv(io.StringIO(raw.decode("utf-8", errors="replace")))

# Airport ID lookup — format: "City, ST: Airport Name"
raw_airport = fetch_bts_lookup(BTS_LOOKUP_URLS["airport"])
raw_airport.columns = ["Code", "Description"]
raw_airport["Code"] = pd.to_numeric(raw_airport["Code"], errors="coerce")
raw_airport = raw_airport.dropna(subset=["Code"])
raw_airport["Code"] = raw_airport["Code"].astype(int)
raw_airport["State"]     = raw_airport["Description"].str.extract(r",\s*([A-Z]{2}):")
raw_airport["CityState"] = raw_airport["Description"].str.extract(r"^(.+):")
airport_lookup = raw_airport[["Code", "State", "CityState"]].dropna(subset=["State"])
print(f"Airport lookup: {len(airport_lookup)} entries")

# City market lookup — format: "City, ST"
raw_city = fetch_bts_lookup(BTS_LOOKUP_URLS["city_market"])
raw_city.columns = ["Code", "Description"]
raw_city["Code"] = pd.to_numeric(raw_city["Code"], errors="coerce")
raw_city = raw_city.dropna(subset=["Code"])
raw_city["Code"] = raw_city["Code"].astype(int)
raw_city["State"]    = raw_city["Description"].str.extract(r",\s*([A-Z]{2})$")
raw_city["CityName"] = raw_city["Description"].str.extract(r"^(.+),\s*[A-Z]{2}$")
city_lookup = raw_city[["Code", "CityName", "State"]].dropna(subset=["State"])
print(f"City market lookup: {len(city_lookup)} entries")

# Carrier lookup
raw_carrier = fetch_bts_lookup(BTS_LOOKUP_URLS["carrier"])
raw_carrier.columns = ["Code", "Description"]
raw_carrier = raw_carrier.dropna()
raw_carrier["Code"]        = raw_carrier["Code"].astype(str).str.strip()
raw_carrier["AirlineName"] = raw_carrier["Description"].str.strip()
carrier_lookup = raw_carrier[["Code", "AirlineName"]]
print(f"Carrier lookup: {len(carrier_lookup)} entries")

# Cell 3 — Load Flight Data and Decode Coded Fields

Loads the raw CSV and renames BTS column names to shorter internal names.
Immediately joins all three lookup tables to add human-readable fields
alongside the numeric IDs. Both forms are retained: numeric IDs are used
by the model for encoding, while the decoded labels are used in output.

In [ ]:
# ===== Cell 3: Load Flight Data and Decode =====

KEEP_RAW = [
    "FL_DATE", "OP_UNIQUE_CARRIER",
    "ORIGIN_AIRPORT_ID", "ORIGIN_CITY_MARKET_ID",
    "DEST_AIRPORT_ID",   "DEST_CITY_MARKET_ID",
    "CRS_DEP_TIME", "DEP_TIME", "DEP_DELAY_NEW", "ARR_DELAY_NEW",
    "DISTANCE", "CARRIER_DELAY", "WEATHER_DELAY",
    "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY",
]

BTS_RENAME = {
    "FL_DATE":               "FlightDate",
    "OP_UNIQUE_CARRIER":     "Airline_Code",
    "ORIGIN_AIRPORT_ID":     "Dep_Airport_ID",
    "ORIGIN_CITY_MARKET_ID": "Dep_Market_ID",
    "DEST_AIRPORT_ID":       "Arr_Airport_ID",
    "DEST_CITY_MARKET_ID":   "Arr_Market_ID",
    "CRS_DEP_TIME":          "Sched_Dep_Time",
    "DEP_TIME":              "Dep_Time",
    "DEP_DELAY_NEW":         "Dep_Delay",
    "ARR_DELAY_NEW":         "Arr_Delay",
    "DISTANCE":              "Distance",
    "CARRIER_DELAY":         "Carrier_Delay",
    "WEATHER_DELAY":         "Weather_Delay",
    "NAS_DELAY":             "NAS_Delay",
    "SECURITY_DELAY":        "Security_Delay",
    "LATE_AIRCRAFT_DELAY":   "LateAircraft_Delay",
}

csv_files = sorted([
    os.path.join(dataset_path, f)
    for f in os.listdir(dataset_path) if f.endswith(".csv")
])

print(f"Found {len(csv_files)} CSV file(s). Loading...")
chunks = []
for fp in csv_files:
    header = pd.read_csv(fp, nrows=0).columns.tolist()
    cols_to_read = [c for c in KEEP_RAW if c in header]
    df = pd.read_csv(fp, usecols=cols_to_read, low_memory=False)
    df.rename(columns=BTS_RENAME, inplace=True)
    df["FlightDate"] = pd.to_datetime(df["FlightDate"], format="mixed")
    chunks.append(df)
    print(f"  Loaded {fp.split('/')[-1]:50s}  rows={len(df):,}")

flights = pd.concat(chunks, ignore_index=True)
print(f"\nTotal rows loaded: {len(flights):,}")

# Decode airport IDs
flights = flights.merge(
    airport_lookup.rename(columns={"Code": "Dep_Airport_ID", "State": "Dep_State", "CityState": "Dep_CityState"}),
    on="Dep_Airport_ID", how="left"
)
flights = flights.merge(
    airport_lookup.rename(columns={"Code": "Arr_Airport_ID", "State": "Arr_State", "CityState": "Arr_CityState"}),
    on="Arr_Airport_ID", how="left"
)

# Decode city market IDs (backup state source)
flights = flights.merge(
    city_lookup.rename(columns={"Code": "Dep_Market_ID", "CityName": "Dep_CityName", "State": "Dep_Market_State"}),
    on="Dep_Market_ID", how="left"
)
flights["Dep_State"] = flights["Dep_State"].fillna(flights["Dep_Market_State"])

# Decode carrier codes
flights = flights.merge(
    carrier_lookup.rename(columns={"Code": "Airline_Code", "AirlineName": "Airline_Name"}),
    on="Airline_Code", how="left"
)

print(f"\nDecoded flights shape: {flights.shape}")
print("\nSample decoded rows:")
print(flights[
    ["FlightDate", "Airline_Code", "Airline_Name",
     "Dep_Airport_ID", "Dep_CityState", "Dep_State", "Dep_Delay"]
].dropna(subset=["Airline_Name", "Dep_CityState"]).head(5).to_string(index=False))

# Cell 4 — Create Target Variables and Class Weights

Drops rows with no recorded departure delay (cancelled or diverted flights).
Creates the binary classification target: 0 for on-time or minor delays
under 15 minutes, 1 for delays of 15 minutes or more. Computes inverse-frequency
class weights to compensate for the imbalanced class distribution (~82% on time).

In [ ]:
# ===== Cell 4: Target Variables and Class Weights =====

flights = flights.dropna(subset=["Dep_Delay"])
flights["Delay_Class"] = np.where(flights["Dep_Delay"] >= 15, 1, 0)

delay_counts = flights["Delay_Class"].value_counts()
delay_pct    = flights["Delay_Class"].value_counts(normalize=True) * 100

print("Delay Class Counts:");  print(delay_counts)
print("\nDelay Class %:");      print(delay_pct.round(2))

total    = len(flights)
weight_0 = total / (2 * delay_counts[0])
weight_1 = total / (2 * delay_counts[1])
class_weights = {0: weight_0, 1: weight_1}
print("\nClass weights:", class_weights)

# Cell 5 — State Mapping

Tags every flight with the U.S. state of its departure airport using the
decoded `Dep_State` field from Cell 3. Also builds a lookup of all valid
state abbreviations present in the dataset, which is used in Cell 9 to
validate user input.

In [ ]:
# ===== Cell 5: State Mapping =====

# Full state name lookup for display purposes
STATE_NAMES = {
    "AL": "Alabama",       "AK": "Alaska",        "AZ": "Arizona",
    "AR": "Arkansas",      "CA": "California",    "CO": "Colorado",
    "CT": "Connecticut",   "DE": "Delaware",       "FL": "Florida",
    "GA": "Georgia",       "HI": "Hawaii",         "ID": "Idaho",
    "IL": "Illinois",      "IN": "Indiana",        "IA": "Iowa",
    "KS": "Kansas",        "KY": "Kentucky",       "LA": "Louisiana",
    "ME": "Maine",         "MD": "Maryland",       "MA": "Massachusetts",
    "MI": "Michigan",      "MN": "Minnesota",      "MS": "Mississippi",
    "MO": "Missouri",      "MT": "Montana",        "NE": "Nebraska",
    "NV": "Nevada",        "NH": "New Hampshire",  "NJ": "New Jersey",
    "NM": "New Mexico",    "NY": "New York",       "NC": "North Carolina",
    "ND": "North Dakota",  "OH": "Ohio",           "OK": "Oklahoma",
    "OR": "Oregon",        "PA": "Pennsylvania",   "RI": "Rhode Island",
    "SC": "South Carolina","SD": "South Dakota",   "TN": "Tennessee",
    "TX": "Texas",         "UT": "Utah",           "VT": "Vermont",
    "VA": "Virginia",      "WA": "Washington",     "WV": "West Virginia",
    "WI": "Wisconsin",     "WY": "Wyoming",
}

# States present in the dataset
valid_states = sorted(flights["Dep_State"].dropna().unique().tolist())
valid_states = [s for s in valid_states if s in STATE_NAMES]  # exclude territories

print(f"States with flight data: {len(valid_states)}")
print(flights["Dep_State"].value_counts().head(10))
print(f"\nUnmapped (territories etc.): {flights['Dep_State'].isna().sum():,}")

# Cell 6 — Feature Engineering

Extracts temporal features from the flight date: year, month, day of week,
weekend flag, and a holiday proximity flag that marks flights within 2 days
of a major U.S. holiday across all years 2014–2024. Adds a departure hour
feature from the scheduled departure time. Fills the delay cause columns
(weather, carrier, NAS, late aircraft) with 0 for on-time flights where
these fields are null. Encodes the airline code and airport ID as integers
for the Random Forest model.

In [ ]:
# ===== Cell 6: Feature Engineering =====
from sklearn.preprocessing import LabelEncoder

# Holiday proximity flag (2014–2024)
holiday_dates = []
for year in range(2014, 2025):
    holiday_dates += [f"{year}-01-01", f"{year}-07-04", f"{year}-12-25"]

approx_floating = [
    "2014-01-20","2014-02-17","2014-05-26","2014-09-01","2014-11-27",
    "2015-01-19","2015-02-16","2015-05-25","2015-09-07","2015-11-26",
    "2016-01-18","2016-02-15","2016-05-30","2016-09-05","2016-11-24",
    "2017-01-16","2017-02-20","2017-05-29","2017-09-04","2017-11-23",
    "2018-01-15","2018-02-19","2018-05-28","2018-09-03","2018-11-22",
    "2019-01-21","2019-02-18","2019-05-27","2019-09-02","2019-11-28",
    "2020-01-20","2020-02-17","2020-05-25","2020-09-07","2020-11-26",
    "2021-01-18","2021-02-15","2021-05-31","2021-09-06","2021-11-25",
    "2022-01-17","2022-02-21","2022-05-30","2022-09-05","2022-11-24",
    "2023-01-16","2023-02-20","2023-05-29","2023-09-04","2023-11-23",
    "2024-01-15","2024-02-19","2024-05-27","2024-09-02","2024-11-28",
]
us_holidays = pd.to_datetime(holiday_dates + approx_floating)

def is_near_holiday(date_series, holidays, window=2):
    flags = np.zeros(len(date_series), dtype=int)
    for h in holidays:
        diff = (date_series - h).dt.days.abs()
        flags |= (diff <= window).astype(int).values
    return flags

flights["Year"]      = flights["FlightDate"].dt.year
flights["Month"]     = flights["FlightDate"].dt.month
flights["DayOfWeek"] = flights["FlightDate"].dt.dayofweek
flights["IsWeekend"] = (flights["DayOfWeek"] >= 5).astype(int)
flights["IsHoliday"] = is_near_holiday(flights["FlightDate"], us_holidays)
flights["Dep_Hour"]  = (flights["Sched_Dep_Time"] // 100).clip(0, 23)

for col in ["Weather_Delay", "Carrier_Delay", "NAS_Delay", "LateAircraft_Delay"]:
    flights[col] = flights[col].fillna(0)

FEATURE_COLS = [
    "Airline_Code", "Dep_Airport_ID", "Year", "Month", "DayOfWeek",
    "IsWeekend", "IsHoliday", "Dep_Hour",
    "Weather_Delay", "Carrier_Delay", "NAS_Delay", "LateAircraft_Delay",
]

model_df = flights[
    FEATURE_COLS + ["Delay_Class", "Dep_State", "Airline_Name", "Dep_CityState"]
].dropna(subset=FEATURE_COLS + ["Delay_Class", "Dep_State"])

le_airline = LabelEncoder()
le_airport = LabelEncoder()

model_df = model_df.copy()
model_df["Airline_enc"]     = le_airline.fit_transform(model_df["Airline_Code"])
model_df["Dep_Airport_enc"] = le_airport.fit_transform(model_df["Dep_Airport_ID"])

FINAL_FEATURES = [
    "Airline_enc", "Dep_Airport_enc", "Year", "Month", "DayOfWeek",
    "IsWeekend", "IsHoliday", "Dep_Hour",
    "Weather_Delay", "Carrier_Delay", "NAS_Delay", "LateAircraft_Delay",
]

X = model_df[FINAL_FEATURES]
y = model_df["Delay_Class"]

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts(normalize=True).round(3)}")

# Cell 7 — Train Random Forest

Stratifies and samples 3 million rows from the full feature matrix for training.
Splits into 80% train / 20% test, computes class weights dynamically from the
sample distribution, and trains a Random Forest with 200 trees. Prints a full
classification report on the held-out test set.

In [ ]:
# ===== Cell 7: Train Random Forest =====
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

SAMPLE_SIZE = 3_000_000

sample_df = model_df.sample(n=min(SAMPLE_SIZE, len(model_df)), random_state=42)
X_s = sample_df[FINAL_FEATURES]
y_s = sample_df["Delay_Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X_s, y_s, test_size=0.2, random_state=42, stratify=y_s
)

n0 = (y_s == 0).sum()
n1 = (y_s == 1).sum()
dynamic_weights = {0: len(y_s) / (2 * n0), 1: len(y_s) / (2 * n1)}
print("Dynamic class weights:", dynamic_weights)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=50,
    class_weight=dynamic_weights,
    n_jobs=-1,
    random_state=42,
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["On Time", "Delayed"]))

# Cell 8 — Save Model and Encoders

Saves the trained Random Forest model and both label encoders to disk using
joblib so the predictor can be loaded and used in other environments without
retraining. Files are saved to `/content/` and can be downloaded from the
Colab file browser.

In [ ]:
# ===== Cell 8: Save Model and Encoders =====
import joblib

SAVE_DIR = "/content/model"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(rf,          f"{SAVE_DIR}/random_forest.joblib")
joblib.dump(le_airline,  f"{SAVE_DIR}/le_airline.joblib")
joblib.dump(le_airport,  f"{SAVE_DIR}/le_airport.joblib")

# Save the feature list and valid states so a loader knows what the model expects
import json
with open(f"{SAVE_DIR}/model_meta.json", "w") as f:
    json.dump({
        "final_features": FINAL_FEATURES,
        "valid_states":   valid_states,
    }, f, indent=2)

print("Saved:")
for fname in os.listdir(SAVE_DIR):
    size_kb = os.path.getsize(os.path.join(SAVE_DIR, fname)) / 1e3
    print(f"  {fname:35s}  {size_kb:8.1f} KB")

print("\nTo load the model elsewhere:")
print("  import joblib")
print("  rf         = joblib.load('random_forest.joblib')")
print("  le_airline = joblib.load('le_airline.joblib')")
print("  le_airport = joblib.load('le_airport.joblib')")

# Cell 9 — Recommender Function

Defines the recommender that accepts a U.S. state abbreviation, a departure
date, and an optional departure hour. It evaluates every airport and airline
combination that historically operated in that state, builds a feature row
for each using the user's date and the airport's median historical departure
hour, and predicts the delay probability for each combination. Returns the
top 5 with the lowest delay probability, with fully decoded human-readable
airport city labels and airline names. The delay cause columns are set to 0
since they are unknown for future flights.

In [ ]:
# ===== Cell 9: Recommender Function =====

def get_top5_recommendations(state, date_str, dep_hour=None):
    """
    Given a U.S. state abbreviation (e.g. 'WA'), a date string (YYYY-MM-DD),
    and an optional departure hour (0-23), return the top 5 airport+airline
    combinations with the lowest predicted delay probability.
    """
    date = pd.to_datetime(date_str)

    year       = date.year
    month      = date.month
    dow        = date.dayofweek
    is_weekend = int(dow >= 5)
    is_holiday = int(any(abs((date - h).days) <= 2 for h in us_holidays))

    # All airport+airline combos that operated in this state
    combos = (
        model_df[model_df["Dep_State"] == state]
        [["Dep_Airport_ID", "Airline_Code", "Airline_Name", "Dep_CityState"]]
        .drop_duplicates(subset=["Dep_Airport_ID", "Airline_Code"])
        .reset_index(drop=True)
    )

    if combos.empty:
        print(f"No flight combinations found for state: {state}")
        return None

    # Median departure hour per airport from historical data
    airport_medians = (
        model_df[model_df["Dep_State"] == state]
        .groupby("Dep_Airport_ID")[["Dep_Hour"]]
        .median()
        .reset_index()
    )
    combos = combos.merge(airport_medians, on="Dep_Airport_ID", how="left")

    if dep_hour is not None:
        combos["Dep_Hour"] = int(dep_hour)

    combos["Year"]      = year
    combos["Month"]     = month
    combos["DayOfWeek"] = dow
    combos["IsWeekend"] = is_weekend
    combos["IsHoliday"] = is_holiday

    # Delay cause columns set to 0 — unknown for future flights
    combos["Weather_Delay"]      = 0
    combos["Carrier_Delay"]      = 0
    combos["NAS_Delay"]          = 0
    combos["LateAircraft_Delay"] = 0

    def safe_encode(encoder, values):
        known = set(encoder.classes_)
        return np.array([
            encoder.transform([v])[0] if v in known else -1
            for v in values
        ])

    combos["Airline_enc"]     = safe_encode(le_airline, combos["Airline_Code"])
    combos["Dep_Airport_enc"] = safe_encode(le_airport, combos["Dep_Airport_ID"])

    combos = combos[
        (combos["Airline_enc"]     != -1) &
        (combos["Dep_Airport_enc"] != -1)
    ]

    X_pred = combos[FINAL_FEATURES].values
    delay_probs = rf.predict_proba(X_pred)[:, 1]

    combos["Delay_Probability"]  = delay_probs
    combos["OnTime_Probability"] = 1 - delay_probs

    top5 = (
        combos[["Dep_CityState", "Airline_Name", "Delay_Probability", "OnTime_Probability"]]
        .sort_values("Delay_Probability")
        .head(5)
        .reset_index(drop=True)
    )
    top5.index += 1
    top5["Delay_Probability"]  = top5["Delay_Probability"].map("{:.1%}".format)
    top5["OnTime_Probability"] = top5["OnTime_Probability"].map("{:.1%}".format)
    top5.columns = ["Airport (City, State)", "Airline",
                    "Delay Probability", "On-Time Probability"]
    return top5

# Cell 10 — User Interaction

Prompts the user for a U.S. state abbreviation, a departure date, and an
optional preferred departure hour. Validates the state against the list of
states present in the dataset, then calls the recommender and prints the
top 5 results.

In [ ]:
# ===== Cell 10: User Interaction =====

print("=== Flight Delay Predictor (2014-2024 dataset) ===")
print(f"Valid states: {', '.join(valid_states)}\n")

state_input = input("Enter your departure state (e.g. WA): ").strip().upper()
date_input  = input("Enter your departure date (YYYY-MM-DD): ").strip()
hour_raw    = input("Enter preferred departure hour 0-23 (or press Enter for median): ").strip()
hour_input  = int(hour_raw) if hour_raw.isdigit() else None

if state_input not in valid_states:
    print(f"Invalid state '{state_input}'. Please choose from: {', '.join(valid_states)}")
else:
    try:
        results = get_top5_recommendations(state_input, date_input, dep_hour=hour_input)
        if results is not None:
            state_name = STATE_NAMES.get(state_input, state_input)
            hour_str   = f" at hour {hour_input}" if hour_input is not None else ""
            print(f"\nTop 5 recommendations for {state_name} on {date_input}{hour_str}:\n")
            print(results.to_string())
    except Exception as e:
        print(f"Something went wrong: {e}")